In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [0]:
spark = SparkSession.builder \
    .appName("Assignment_4_Data_Ingestion") \
    .getOrCreate()

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/assignment_data/adult.csv",
    header=True,
    inferSchema=True
)

display(df.limit(5))
df.printSchema() #validate the schema

age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


root
 |-- age: integer (nullable = true)
 |-- workclass: string (nullable = true)
 |-- fnlwgt: integer (nullable = true)
 |-- education: string (nullable = true)
 |-- education.num: integer (nullable = true)
 |-- marital.status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- relationship: string (nullable = true)
 |-- race: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- capital.gain: integer (nullable = true)
 |-- capital.loss: integer (nullable = true)
 |-- hours.per.week: integer (nullable = true)
 |-- native.country: string (nullable = true)
 |-- income: string (nullable = true)



In [0]:
df = df.toDF(*[
    c.replace(".", "_")
     .replace("-", "_")
     .replace(" ", "_")
    for c in df.columns
])

In [0]:
df = df.filter(col("workclass") != "?")
df = df.filter(col("occupation") != "?")
df = df.filter(col("native_country") != "?")

df = df.dropDuplicates()
df = df.na.drop()

## Add Ingestion Timestamp

In [0]:
df = df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

In [0]:
# Add source metadata

df = df.withColumn(
    "source_file",
    lit("adult.csv")
)

In [0]:
# verifying 

display(
    df.select(
        "ingestion_timestamp",
        "source_file"
    ).limit(5)
)

ingestion_timestamp,source_file
2026-07-12T20:13:40.421Z,adult.csv
2026-07-12T20:13:40.421Z,adult.csv
2026-07-12T20:13:40.421Z,adult.csv
2026-07-12T20:13:40.421Z,adult.csv
2026-07-12T20:13:40.421Z,adult.csv


## Feature Engineering

In [0]:
categorical_cols = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country"
]

for column in categorical_cols:

    indexer = StringIndexer(
        inputCol=column,
        outputCol=column+"_index",
        handleInvalid="keep"
    )

    df = indexer.fit(df).transform(df)

## Vector Assembler

In [0]:
feature_columns = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "workclass_index",
    "education_index",
    "marital_status_index",
    "occupation_index",
    "relationship_index",
    "race_index",
    "sex_index",
    "native_country_index"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df = assembler.transform(df)

In [0]:
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features"
)

scaler_model = scaler.fit(df)

df = scaler_model.transform(df)

In [0]:
# Save as Parquet

df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/assignment_data/ingested_data.parquet"
)

In [0]:
# Read Parquet

processed_df = spark.read.parquet(
    "/Volumes/workspace/default/assignment_data/ingested_data.parquet"
)

In [0]:
# Encode the label

label_indexer = StringIndexer(
    inputCol="income",
    outputCol="label"
)

processed_df = label_indexer.fit(processed_df).transform(processed_df)

In [0]:
# Train test split

train, test = processed_df.randomSplit(
    [0.8,0.2],
    seed=42
)

## Decision Tree

In [0]:
dt = DecisionTreeClassifier(
    featuresCol="scaled_features",
    labelCol="label"
)

model = dt.fit(train)

In [0]:
# Prediction

predictions = model.transform(test)

In [0]:
# Accuracy

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)

print("Accuracy:", accuracy)

Accuracy: 0.8418069678279714


In [0]:
# Show Prediction

display(
    predictions.select(
        "label",
        "prediction",
        "probability"
    ).limit(10)
)

label,prediction,probability
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
